# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Refresh / Content Opportunity Scoring :
# Rather than just binary classification (predicting simply if a page is decaying or not),
# content managers have limited time and require a prioritized queue. We predict the probability that a page is declining in search visibility and
# combine it with demand signals to produce a normalized opportunity score (0 to 100) that ranks pages by review priority.

import pandas as pd
import os

# Load dataset safely from relative path
path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "content_refresh_anonymized.csv"
df = pd.read_csv(path)

# Check class distribution of the binary status
declining_counts = df['trend_direction'].value_counts()
declining_pcts = df['trend_direction'].value_counts(normalize=True) * 100

print("--- Class Breakdown for Scoring Task ---")
for trend, count in declining_counts.items():
    print(f"Trend '{trend}': {count:,} pages ({declining_pcts[trend]:.2f}%)")

--- Class Breakdown for Scoring Task ---
Trend 'down': 16,262 pages (54.21%)
Trend 'stable': 5,962 pages (19.87%)
Trend 'up': 4,388 pages (14.63%)
Trend 'new': 2,236 pages (7.45%)
Trend 'flat': 1,152 pages (3.84%)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Binary Proxy Label: is_declining = (trend_direction == "down")
# This is a Proxy Label derived from a defined heuristic rule in the starter dataset (comparing search/session trends across recent windows).
# It measures historical performance drop in the current window rather than an observed future post-refresh outcome.
# In a full capstone setup, a stronger target would predict future traffic loss over a subsequent 30-day window using features from the prior 90 days to prevent target leakage

# Define the proxy binary label
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

total_rows = len(df)
positive_labels = df['is_declining'].sum()
positive_ratio = (positive_labels / total_rows) * 100

print(f"Total rows evaluated: {total_rows:,}")
print(f"Positive target instances (is_declining == 1): {positive_labels:,} ({positive_ratio:.2f}%)")

Total rows evaluated: 30,000
Positive target instances (is_declining == 1): 16,262 (54.21%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# primary Metric: Precision@20 (supported by Average Precision / ROC AUC).
# What number means 'good':A Precision@20 of > 0.70 (70%) is considered good for this task.

import numpy as np

# Compute a simple baseline rule score
top_20 = df.sort_values(by='impressions_90d', ascending=False).head(20)

correct_count = top_20['is_declining'].sum()
precision_at_20 = top_20['is_declining'].mean()

print(f"Baseline Precision@20: {precision_at_20:.2f} ({correct_count} out of 20 correct)")

Baseline Precision@20: 0.45 (9 out of 20 correct)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Unit of Analysis: One row represents a unique pseudonymized content page (content_id) belonging to a client (client_id), aggregated over a fixed 90-day performance window.
# There are no duplicate content IDs in this dataset slice,ensuring that each page appears exactly once in the evaluation matrix.

# Display shape and verify grain uniqueness
shape = df.shape
unique_contents = df['content_id'].nunique()
unique_clients = df['client_id'].nunique()

print(f"Dataframe Shape: {shape[0]:,} rows x {shape[1]} columns")
print(f"Unique Content IDs: {unique_contents:,} (Matches total rows: {unique_contents == shape[0]})")
print(f"Unique Client IDs: {unique_clients}")

# Display the first 3 rows showing grain identifiers and main signals
df[['content_id', 'client_id', 'impressions_90d', 'sessions_90d', 'avg_position', 'trend_direction']].head(3)

Dataframe Shape: 30,000 rows x 45 columns
Unique Content IDs: 30,000 (Matches total rows: True)
Unique Client IDs: 32


,content_id,client_id,impressions_90d,sessions_90d,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#Over-flagging: A simple rule like if trend == 'down' and impressions >= 500 flags 9,961 pages.
# A human team cannot review 10,000 pages,and the rule provides no granular order within those 10,000 pages.

#Smooth Priority Ranking: ML algorithms (like Random Forest or Decision Trees) combine these continuous signals to produce a calibrated probability,
#sorting candidates so the top 20 represent the highest true risk and demand opportunity.

# Demonstrating rule over-flagging vs capacity
simple_rule_matches = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)]
stale_rule_matches = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]

print(f"Pages flagged by 'Demand + Declining' rule: {len(simple_rule_matches):,} pages")
print(f"Pages flagged by 'Stale + Demand' rule: {len(stale_rule_matches):,} pages")
print("Conclusion: Fixed rules either flag far too many pages to be actionable or miss decaying pages completely.")


Pages flagged by 'Demand + Declining' rule: 9,961 pages
Pages flagged by 'Stale + Demand' rule: 17 pages
Conclusion: Fixed rules either flag far too many pages to be actionable or miss decaying pages completely.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.